In [3]:
import pandas as pd
import numpy as np
import os
import pickle
from typing import Dict, Any

# --- 경로 설정 및 모델 저장 위치 수정 ---
# 과제 2의 최종 분석 데이터 (surprise_z 및 수익률 포함)
VENDOR_ANALYSIS_PATH = "../../output/problem2_vendor/problem2_vendor_analysis_base.csv"
OUT_DIR = "../../output/model" # 모델 아티팩트 저장 폴더 (요청에 따라 수정됨)
MODEL_FILE_NAME = "basic_rule_model.pkl"
os.makedirs(OUT_DIR, exist_ok=True) # 폴더가 없으면 생성


# --- 모델 하이퍼파라미터 정의 ---
Z_SCORE_THRESHOLD = 2.0
MOST_SENSITIVE_GICS_SECTORS = [] # 과제 3 제외


# --- 2. 기본 규칙 기반 의사 결정 함수 ---
def get_investment_decision_basic(df: pd.DataFrame) -> pd.DataFrame:
    """
    ARIMA Surprise Z-Score (±2.0)만을 기준으로 매수/매도 결정을 내리는 기본 모델.
    """
    df_result = df.copy()
    
    is_buy_signal = df_result['surprise_z'] > Z_SCORE_THRESHOLD
    is_sell_signal = df_result['surprise_z'] < -Z_SCORE_THRESHOLD
    
    conditions = [
        is_buy_signal,
        is_sell_signal,
        True
    ]
    
    choices = [
        'BUY', 
        'SELL', 
        'HOLD'
    ]
    
    df_result['decision'] = np.select(conditions, choices, default='HOLD')
    return df_result


# --- 3. 모델 실행, 검증 및 저장 ---
print("-> 1. 데이터 로드 및 기본 모델 실행...")

try:
    # 데이터 로드 (과제 2 결과 사용)
    df_analysis = pd.read_csv(
        VENDOR_ANALYSIS_PATH, 
        usecols=['symbol', 'date', 'surprise_z', 'return_post_1d', 'return_post_2d'],
        dtype={'surprise_z': np.float32, 'return_post_1d': np.float32, 'return_post_2d': np.float32}
    ).dropna(subset=['surprise_z', 'return_post_1d'])
    
    # 모델 실행
    df_decision = get_investment_decision_basic(df_analysis)
    
    # 시그널 발생 행만 필터링
    df_signals = df_decision[df_decision['decision'].isin(['BUY', 'SELL'])].copy()
    
    if df_signals.empty:
        print("분석: BUY/SELL 시그널이 발생하지 않아 검증을 수행할 수 없습니다.")
        exit()

    # 4. 모델 성능 검증 (수익률 요약)
    df_summary = df_signals.groupby('decision')[['return_post_1d', 'return_post_2d']].mean().mul(100).round(4)
    df_summary = df_summary.rename(columns={'return_post_1d': 'Avg_Return_Post_1D (%)', 'return_post_2d': 'Avg_Return_Post_2D (%)'})
    
    # 5. 모델 규칙 저장 (.pkl)
    model_rules = {
        'model_name': 'Basic_ZScore_Classifier',
        'z_threshold': Z_SCORE_THRESHOLD,
        'sensitive_sectors': MOST_SENSITIVE_GICS_SECTORS, 
        'version': 'v1.0 (No GICS Filter)'
    }
    
    model_path = os.path.join(OUT_DIR, MODEL_FILE_NAME)
    with open(model_path, 'wb') as f:
        pickle.dump(model_rules, f)
        
    print(f"\n[OK] 모델 규칙 저장 완료: {model_path}")
    
    # 6. 결과 출력
    print("\n" + "="*70)
    print("과제 4: 기본 모델 (산업 제외) 수익률 검증 결과")
    print("="*70)
    print(df_summary.to_markdown())
    print("="*70)

except FileNotFoundError as e:
    print(f"❌ 오류: 필요한 파일을 찾을 수 없습니다. 경로를 확인하세요: {e}")
except Exception as e:
    print(f"❌ 오류: 데이터 처리 중 예상치 못한 오류 발생: {e}")

-> 1. 데이터 로드 및 기본 모델 실행...

[OK] 모델 규칙 저장 완료: ../../output/model/basic_rule_model.pkl

과제 4: 기본 모델 (산업 제외) 수익률 검증 결과
| decision   |   Avg_Return_Post_1D (%) |   Avg_Return_Post_2D (%) |
|:-----------|-------------------------:|-------------------------:|
| BUY        |                   0.2718 |                   0.8935 |
| SELL       |                   0.3046 |                   0.7187 |
